# TP Databricks: Análisis de películas (MovieLens)
## Notebook 01: Carga, exploración y transformación
Este notebook resuelve las **Etapas 1, 2 y 3** del trabajo práctico y deja iniciada la **Etapa 4** (guardado en Delta Lake).
- Notebook 02: Etapa 4 (validación en Delta Lake)
- Notebook 03: Etapa 5 (visualización y análisis)

**Dataset:** MovieLens `ml-latest-small` (`movies.csv` y `ratings.csv`).

In [0]:
# Prueba inicial: verificar que Python y Spark funcionan
print("Hola Databricks")
print(spark.version)


Hola Databricks
4.2.0


In [0]:
%sql
SELECT 1 AS prueba

prueba
1


In [0]:
# Ruta base donde están los archivos CSV (catálogo / schema / volume)
ruta = "/Volumes/workspace/tp7_peliculas/peliculas/"

# Lectura del CSV de películas
movies = spark.read.csv(ruta + "movies.csv", header=True, inferSchema=True)
movies.show(5, truncate=False)
movies.printSchema()

+-------+----------------------------------+-------------------------------------------+
|movieId|title                             |genres                                     |
+-------+----------------------------------+-------------------------------------------+
|1      |Toy Story (1995)                  |Adventure|Animation|Children|Comedy|Fantasy|
|2      |Jumanji (1995)                    |Adventure|Children|Fantasy                 |
|3      |Grumpier Old Men (1995)           |Comedy|Romance                             |
|4      |Waiting to Exhale (1995)          |Comedy|Drama|Romance                       |
|5      |Father of the Bride Part II (1995)|Comedy                                     |
+-------+----------------------------------+-------------------------------------------+
only showing top 5 rows
root
 |-- movieId: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- genres: string (nullable = true)



In [0]:
# Lectura del CSV de puntuaciones
ratings = spark.read.csv(ruta + "ratings.csv", header=True, inferSchema=True)
ratings.show(5)
ratings.printSchema()

# Cantidad de filas de cada tabla
print("Películas:", movies.count())
print("Ratings:", ratings.count())

+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|      1|   4.0|964982703|
|     1|      3|   4.0|964981247|
|     1|      6|   4.0|964982224|
|     1|     47|   5.0|964983815|
|     1|     50|   5.0|964982931|
+------+-------+------+---------+
only showing top 5 rows
root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- timestamp: integer (nullable = true)

Películas: 9742
Ratings: 100836


In [0]:
# Crear vistas temporales para poder usar SQL
movies.createOrReplaceTempView("movies")
ratings.createOrReplaceTempView("ratings")

In [0]:
%sql
-- Primeras 10 películas
SELECT * FROM movies LIMIT 10

movieId,title,genres
1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
2,Jumanji (1995),Adventure|Children|Fantasy
3,Grumpier Old Men (1995),Comedy|Romance
4,Waiting to Exhale (1995),Comedy|Drama|Romance
5,Father of the Bride Part II (1995),Comedy
6,Heat (1995),Action|Crime|Thriller
7,Sabrina (1995),Comedy|Romance
8,Tom and Huck (1995),Adventure|Children
9,Sudden Death (1995),Action
10,GoldenEye (1995),Action|Adventure|Thriller


In [0]:
%sql
-- Cuántas puntuaciones, cuántos usuarios y promedio general
SELECT COUNT(*) AS total_ratings,
       COUNT(DISTINCT userId) AS usuarios,
       ROUND(AVG(rating), 2) AS promedio_general,
       MIN(rating) AS minimo,
       MAX(rating) AS maximo
FROM ratings

total_ratings,usuarios,promedio_general,minimo,maximo
100836,610,3.5,0.5,5.0


In [0]:
from pyspark.sql import functions as F

# --- Duplicados ---
print("Movies antes:", movies.count())
movies_limpio = movies.dropDuplicates()
print("Movies después de quitar duplicados:", movies_limpio.count())

print("Ratings antes:", ratings.count())
ratings_limpio = ratings.dropDuplicates()
print("Ratings después de quitar duplicados:", ratings_limpio.count())

# --- Nulos: contar nulos por columna ---
print("Nulos en movies:")
movies_limpio.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in movies_limpio.columns]).show()

print("Nulos en ratings:")
ratings_limpio.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in ratings_limpio.columns]).show()

# Eliminar filas con nulos en columnas clave (si hubiera)
movies_limpio = movies_limpio.dropna(subset=["movieId", "title"])
ratings_limpio = ratings_limpio.dropna(subset=["userId", "movieId", "rating"])

Movies antes: 9742
Movies después de quitar duplicados: 9742
Ratings antes: 100836
Ratings después de quitar duplicados: 100836
Nulos en movies:
+-------+-----+------+
|movieId|title|genres|
+-------+-----+------+
|      0|    0|     0|
+-------+-----+------+

Nulos en ratings:
+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     0|      0|     0|        0|
+------+-------+------+---------+



In [0]:
# Extraer el año del título, por ejemplo "Toy Story (1995)" -> 1995
# Si el título no tiene año, queda null en lugar de dar error
anio_texto = F.regexp_extract("title", r"\((\d{4})\)", 1)

movies_limpio = movies_limpio.withColumn(
    "anio", F.when(anio_texto != "", anio_texto.cast("int"))
)

# Convertir el timestamp (segundos) a fecha y sacar el año de la puntuación
ratings_limpio = ratings_limpio.withColumn(
    "fecha", F.to_date(F.from_unixtime("timestamp"))
).withColumn("anio_rating", F.year("fecha"))

# Columna calculada: clasificar la puntuación
ratings_limpio = ratings_limpio.withColumn(
    "valoracion",
    F.when(F.col("rating") >= 4, "Alta")
     .when(F.col("rating") >= 3, "Media")
     .otherwise("Baja")
)

# Cuántas películas no tienen año en el título (se van a descartar)
print("Películas sin año:", movies_limpio.filter(F.col("anio").isNull()).count())

# Filtro: quedarnos con películas que tengan año válido
movies_limpio = movies_limpio.filter(F.col("anio").isNotNull())

movies_limpio.show(5, truncate=False)
ratings_limpio.show(5)

Películas sin año: 13
+-------+----------------------------------+-------------------------------+----+
|movieId|title                             |genres                         |anio|
+-------+----------------------------------+-------------------------------+----+
|16     |Casino (1995)                     |Crime|Drama                    |1995|
|55     |Georgia (1995)                    |Drama                          |1995|
|60     |Indian in the Cupboard, The (1995)|Adventure|Children|Fantasy     |1995|
|94     |Beautiful Girls (1996)            |Comedy|Drama|Romance           |1996|
|160    |Congo (1995)                      |Action|Adventure|Mystery|Sci-Fi|1995|
+-------+----------------------------------+-------------------------------+----+
only showing top 5 rows
+------+-------+------+---------+----------+-----------+----------+
|userId|movieId|rating|timestamp|     fecha|anio_rating|valoracion|
+------+-------+------+---------+----------+-----------+----------+
|     1|    

In [0]:
# Join de ratings con películas
df = ratings_limpio.join(movies_limpio, on="movieId", how="inner")

# Agregación: promedio y cantidad de puntuaciones por película
ranking = (df.groupBy("movieId", "title", "anio")
             .agg(F.round(F.avg("rating"), 2).alias("promedio"),
                  F.count("rating").alias("cant_ratings"))
          )

# Filtro: solo películas con al menos 50 puntuaciones (para que el promedio sea confiable)
top_peliculas = ranking.filter(F.col("cant_ratings") >= 50).orderBy(F.desc("promedio"))
top_peliculas.show(10, truncate=False)

+-------+---------------------------------------------------------------------------+----+--------+------------+
|movieId|title                                                                      |anio|promedio|cant_ratings|
+-------+---------------------------------------------------------------------------+----+--------+------------+
|318    |Shawshank Redemption, The (1994)                                           |1994|4.43    |317         |
|858    |Godfather, The (1972)                                                      |1972|4.29    |192         |
|750    |Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1964)|1964|4.27    |97          |
|1276   |Cool Hand Luke (1967)                                                      |1967|4.27    |57          |
|2959   |Fight Club (1999)                                                          |1999|4.27    |218         |
|904    |Rear Window (1954)                                                         |1954|4.26  

In [0]:
# ETAPA 4: Guardar los datos transformados en formato Delta Lake
spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA tp7_peliculas")

df.write.format("delta").mode("overwrite").saveAsTable("peliculas_ratings")
top_peliculas.write.format("delta").mode("overwrite").saveAsTable("top_peliculas")

print("Tablas guardadas correctamente")

Tablas guardadas correctamente


In [0]:
# Verificar que las tablas existen
spark.sql("SHOW TABLES IN workspace.tp7_peliculas").show()

+-------------+-----------------+-----------+
|     database|        tableName|isTemporary|
+-------------+-----------------+-----------+
|tp7_peliculas|peliculas_ratings|      false|
|tp7_peliculas|    top_peliculas|      false|
|             |           movies|       true|
|             |          ratings|       true|
+-------------+-----------------+-----------+

